# Volume 11, Chapter 9: Expanders and Ramanujan graphs (Python companion)

Exercises first, solutions at the end. Mixing on expanders against cycles and cubes, the hitting property and the Chernoff bound for expander walks (randomness saving), the Kesten–McKay law, Ihara's zeta function and the prime number theorem for graphs, the Cayley graphs of $\mathrm{SL}_2(\mathbf F_p)$, and a Ramanujan graph of Lubotzky, Phillips and Sarnak built from sums of four squares. Requires `numpy`, `scipy` and `networkx`.

In [1]:
import math
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spl
import networkx as nx
rng = np.random.default_rng(1109)

def adjacency(G):
    return nx.to_numpy_array(G, nodelist=sorted(G.nodes()))

## Exercises

**P9.1 (★) Mixing.** Compare the total variation distance to uniform after $t$ steps of the lazy walk, started at a vertex, on the cycle $C_{1024}$, the hypercube $\{0,1\}^{10}$ and a random $4$-regular graph on $1024$ vertices. How many steps does each need to reach distance $0.01$? Compare with Proposition 9.1.

**P9.2 (★★) The hitting property.** On a random $4$-regular graph with $n=2000$ vertices, let $B$ be a random half of the vertices. Estimate by simulation $P(X_0,\dots,X_t\in B)$, compute it exactly by matrix products, and compare with the bound $\beta(\beta+\alpha)^t$ of Proposition 9.2 and with the value $\beta^{t+1}$ for independent samples.

**P9.3 (★★) Chernoff for walks.** On the same graph, take $f=\mathbf 1_B$ for a set $B$ of density $0.3$. For $t=200$, estimate the tail $P(S_t/t\ge0.3+\varepsilon)$ for walks and for independent samples, for $\varepsilon=0.05,0.1,0.15$, and compare with Theorem 9.3. How many random bits does each method use?

**P9.4 (★) The Kesten–McKay law.** For $d=3,4,10$ draw a random $d$-regular graph on $3000$ vertices, compare the histogram of its eigenvalues with $\nu_d$, and its moments $\frac1n\operatorname{tr}A^k$ with the numbers $a_k$ of closed walks in the tree, computed by dynamic programming on the distance.

**P9.5 (★★) Ihara's zeta function.** For the Petersen graph, build the $30\times30$ matrix $B$, check Theorem 9.9 at a few values of $u$, and check (9.4). Then, for a random $3$-regular graph on $300$ vertices, test whether it is Ramanujan, compute $\pi(k)$ for $k\le40$ by Möbius inversion of $N_k=\sum_{\ell\mid k}\ell\pi(\ell)$, and compare with $\frac{2^k}k$ and the error bound of Theorem 9.11.

**P9.6 (★★★) Cayley graphs of $\mathrm{SL}_2(\mathbf F_p)$ and an LPS graph.** (a) Compute the second eigenvalue of the Cayley graph of $\mathrm{SL}_2(\mathbf F_p)$ with generators $(\begin{smallmatrix}1&\pm1\\0&1\end{smallmatrix})$, $(\begin{smallmatrix}1&0\\\pm1&1\end{smallmatrix})$ for primes $p\le53$ (up to $148\,824$ vertices). Is it Ramanujan? Is there a uniform spectral gap? (b) Build the Lubotzky–Phillips–Sarnak graph $X^{5,13}$: the Cayley graph of $\mathrm{PGL}_2(\mathbf F_{13})$ with the six generators coming from the solutions of $a_0^2+a_1^2+a_2^2+a_3^2=5$ with $a_0>0$ odd, $a_1,a_2,a_3$ even, and check that it is a $6$-regular bipartite Ramanujan graph.

## Solutions

### P9.1

**How to find it.** Propagate the law exactly with the sparse transition matrix of the lazy walk $\frac12(I+P)$ (laziness removes the periodicity of the bipartite cycle and cube).

In [2]:
def lazy(G):
    A = sp.csr_matrix(adjacency(G)); d = np.asarray(A.sum(axis=1)).ravel()
    return 0.5 * (sp.identity(A.shape[0]) + sp.diags(1 / d) @ A)

graphs = {"cycle": nx.cycle_graph(1024),
          "hypercube": nx.convert_node_labels_to_integers(nx.hypercube_graph(10)),
          "random 4-regular": nx.random_regular_graph(4, 1024, seed=1)}
for name, G in graphs.items():
    P = lazy(G).T.tocsr(); n = P.shape[0]
    p = np.zeros(n); p[0] = 1; t = 0
    while 0.5 * np.abs(p - 1 / n).sum() > 0.01 and t < 10**6:
        p = P @ p; t += 1
    lam = np.sort(np.linalg.eigvalsh(adjacency(G)))
    d = adjacency(G)[0].sum(); alpha = (1 + lam[-2] / d) / 2
    bound = math.log(math.sqrt(n) / 0.02) / math.log(1 / alpha)
    print(f"{name:>16}: {t:>6} steps to TV < 0.01;  lazy alpha = {alpha:.6f}, bound of Prop 9.1: {bound:.0f}")

           cycle: 441289 steps to TV < 0.01;  lazy alpha = 0.999991, bound of Prop 9.1: 783834
       hypercube:     46 steps to TV < 0.01;  lazy alpha = 0.900000, bound of Prop 9.1: 70
random 4-regular:     62 steps to TV < 0.01;  lazy alpha = 0.929105, bound of Prop 9.1: 100


The cycle needs about $441\,000\approx0.42\,n^2$ steps, the cube $46$ and the random graph $62$, and the bound of Proposition 9.1 is within a factor $2$ for the last two. At this size the cube even beats the expander: $\log n\log\log n$ and $\log n$ differ only by the factor $\log\log n$, and the lazy walk halves the spectral gap of the random graph ($1-\alpha\approx0.07$). The difference appears as $n$ grows: the gap of the cube $\{0,1\}^k$ is $\frac1{2k}\to0$ (here $k=10$ and $\alpha=0.9$), while that of the random $4$-regular graph stays near $\frac{4-2\sqrt3}{8}\approx0.067$ for every $n$ (Friedman's theorem).

### P9.2

**How to find it.** $P(X_0,\dots,X_t\in B)=\frac1n\mathbf 1_B^{\mathsf T}(\Pi P\Pi)^t\mathbf 1_B$ exactly; the simulation runs many walks in parallel with a neighbour table.

In [3]:
n, d = 2000, 4
G = nx.random_regular_graph(d, n, seed=2)
A = adjacency(G); ev = np.sort(np.linalg.eigvalsh(A))
alpha = max(ev[-2], -ev[0]) / d
nbr = np.array([sorted(G[x]) for x in range(n)])
B = np.zeros(n, bool); B[rng.choice(n, n // 2, replace=False)] = True
beta = B.mean()
P = sp.csr_matrix(A / d); v = B.astype(float) / n
walks = rng.integers(n, size=400_000); alive = B[walks]
Pi = sp.diags(B.astype(float)); rate = spl.eigsh(Pi @ P @ Pi, k=1, which='LA', return_eigenvectors=False)[0]
print(f"alpha = {alpha:.4f}, beta = {beta}, largest eigenvalue of Pi P Pi = {rate:.4f}")
for t in range(1, 13):
    v = (P @ v) * B
    walks = nbr[walks, rng.integers(d, size=walks.size)]; alive &= B[walks]
    if t % 3 == 0:
        print(f"t = {t:>2}: exact {v.sum():.2e}   simulated {alive.mean():.2e}   bound {beta * (beta + alpha)**t:.2e}   independent {beta**(t + 1):.2e}")

alpha = 0.8641, beta = 0.5, largest eigenvalue of Pi P Pi = 0.7053
t =  3: exact 9.98e-02   simulated 9.94e-02   bound 1.27e+00   independent 6.25e-02
t =  6: exact 2.91e-02   simulated 2.92e-02   bound 3.22e+00   independent 7.81e-03
t =  9: exact 8.99e-03   simulated 9.01e-03   bound 8.17e+00   independent 9.77e-04
t = 12: exact 2.89e-03   simulated 2.88e-03   bound 2.07e+01   independent 1.22e-04


The simulation agrees with the exact value, which decays geometrically, by a factor of about $0.68$ per step over these times, approaching $0.71$, the top eigenvalue of $\Pi P\Pi$: slower than the rate $\beta=\frac12$ of independent samples, but still exponential. With $d=4$, $\alpha\approx0.86$ and $\beta+\alpha>1$, so the bound of Proposition 9.2 says nothing here; it must hold for every set $B$, including sets with few edges leaving them, and it becomes useful once $\alpha<1-\beta$. An amplification as in Exercise 9.2 would use a graph of larger degree, or walk with steps of the power $A^j$ (whose $\alpha$ is raised to the power $j$), to get $\alpha\le\frac14$.

### P9.3

**How to find it.** Count the visits to $B$ along $t=200$ steps for many walks, and compare with binomial samples.

In [4]:
B3 = np.zeros(n, bool); B3[rng.choice(n, int(0.3 * n), replace=False)] = True
t, runs = 200, 100_000
walks = rng.integers(n, size=runs); S = np.zeros(runs)
for _ in range(t):
    walks = nbr[walks, rng.integers(d, size=runs)]; S += B3[walks]
indep = rng.binomial(t, 0.3, size=runs)
for eps in (0.05, 0.1, 0.15):
    print(f"eps = {eps:.2f}: walk {np.mean(S / t >= 0.3 + eps):.5f}   independent {np.mean(indep / t >= 0.3 + eps):.5f}   "
          f"Theorem 9.3 bound {math.exp(-(1 - alpha) * eps**2 * t / 20):.3f}")
print(f"standard deviation of S/t: walk {np.std(S / t):.4f}, independent {np.std(indep / t):.4f}")
print(f"random bits: walk {math.log2(n) + t * math.log2(d):.0f}, independent {t * math.log2(n):.0f}")

eps = 0.05: walk 0.14769   independent 0.07351   Theorem 9.3 bound 0.997
eps = 0.10: walk 0.01855   independent 0.00176   Theorem 9.3 bound 0.986
eps = 0.15: walk 0.00108   independent 0.00000   Theorem 9.3 bound 0.970
standard deviation of S/t: walk 0.0455, independent 0.0324
random bits: walk 411, independent 2193


The walk's average fluctuates more than independent samples (standard deviation $1.4$ times larger, the price of the correlations between consecutive steps; the worst case over all sets $B$ is a factor $\sqrt{(1+\alpha)/(1-\alpha)}\approx3.7$), so its tails are fatter, but they still decay like a Gaussian. Theorem 9.3 is true but says nothing at this size: its constant $20$ and the factor $1-\alpha\approx0.14$ make it useful only for walks of tens of thousands of steps. Its point is the guarantee, uniform in $n$ and in $f$, for $411$ random bits instead of $2193$.

### P9.4

**How to find it.** The distance from the root on $T_d$ is a Markov chain: from $0$ to $1$ in $d$ ways, from $j\ge1$ up in $d-1$ ways and down in one. Count walks with it, exactly in integers.

In [5]:
def tree_closed_walks(d, K):
    v = [1] + [0] * (K + 1); out = [1]
    for _ in range(K):
        w = [0] * (K + 2); w[1] += d * v[0]
        for j in range(1, K + 1):
            w[j + 1] += (d - 1) * v[j]; w[j - 1] += v[j]
        v = w; out.append(v[0])
    return out

def kesten_mckay(d, x):
    return d * np.sqrt(np.clip(4 * (d - 1) - x**2, 0, None)) / (2 * np.pi * (d**2 - x**2))

for d in (3, 4, 10):
    G = nx.random_regular_graph(d, 3000, seed=d); ev = np.linalg.eigvalsh(adjacency(G))
    r = 2 * math.sqrt(d - 1); edges = np.linspace(-r, r, 13)
    h, _ = np.histogram(ev, bins=edges); mids = (edges[1:] + edges[:-1]) / 2
    print(f"d = {d}: histogram / density at 6 bin centres:",
          np.round(h[::2] / (3000 * (edges[1] - edges[0])) / kesten_mckay(d, mids[::2]), 2))
    a = tree_closed_walks(d, 10)
    print("   moments (1/n) tr A^k:", [round(float(np.mean(ev**k)), 1) for k in (2, 4, 6, 8, 10)])
    print("   tree a_k            :", [a[k] for k in (2, 4, 6, 8, 10)])

d = 3: histogram / density at 6 bin centres: [0.93 1.01 0.99 1.01 1.   1.  ]
   moments (1/n) tr A^k: [3.0, 15.0, 87.1, 544.1, 3555.6]
   tree a_k            : [3, 15, 87, 543, 3543]


d = 4: histogram / density at 6 bin centres: [0.91 1.01 0.99 0.99 1.01 1.  ]
   moments (1/n) tr A^k: [4.0, 28.0, 232.9, 2110.1, 20183.6]
   tree a_k            : [4, 28, 232, 2092, 19864]


d = 10: histogram / density at 6 bin centres: [0.91 1.   1.   1.   1.   0.99]
   moments (1/n) tr A^k: [10.0, 192.1, 4709.4, 145774.9, 6409904.4]
   tree a_k            : [10, 190, 4420, 113950, 3128140]


The ratios histogram/density are within a few percent of $1$, except in the extreme bins, where the density varies fast inside the bin and the edge eigenvalues fluctuate. The moments agree with the tree, $a_2=d$, $a_4=d(2d-1)$, $a_6=d(5d^2-6d+2)$ (Exercise 9.3), up to two corrections: the trivial eigenvalue $d$, which adds $d^k/n$ (for $d=10$ and $k=8$ that is $10^8/3000\approx33\,000$, almost all of the visible gap), and the few short cycles (a random $3$-regular graph has on average $\frac{2^3}{6}\approx1.3$ triangles, Bollobás).

### P9.5

**How to find it.** Index the oriented edges, fill $B$ from the definition, and compare $\det(I-uB)$ with $(1-u^2)^{m-n}\det(I-uA+qu^2I)$. For $\pi(k)$, $k\pi(k)=\sum_{\ell\mid k}\mu(k/\ell)N_\ell$ with the Möbius function, and $N_\ell$ from (9.4) in exact integer arithmetic (the sums $\mu_j^\ell+\mu_j'^\ell$ are computed from the eigenvalues; rounding $N_\ell$ to the nearest integer is safe because the error is far below $\frac12$ at this size).

In [6]:
def hashimoto(G):
    oe = [(x, y) for x, y in G.edges()] + [(y, x) for x, y in G.edges()]
    index = {e: i for i, e in enumerate(oe)}
    Bm = np.zeros((len(oe), len(oe)))
    for (x, y), i in index.items():
        for z in G[y]:
            if z != x: Bm[i, index[(y, z)]] = 1
    return Bm

G = nx.petersen_graph(); A = adjacency(G); Bm = hashimoto(G)
n, m, q = 10, 15, 2
for u in (0.1, 0.3, -0.45, 0.6):
    lhs = np.linalg.det(np.eye(2 * m) - u * Bm)
    rhs = (1 - u * u)**(m - n) * np.linalg.det(np.eye(n) - u * A + q * u * u * np.eye(n))
    print(f"u = {u:>5}: det(I - uB) = {lhs:.10f}   Bass {rhs:.10f}")
lam = np.linalg.eigvalsh(A)
roots = np.concatenate([np.roots([1, -l, q]) for l in lam])
for k in range(1, 11):
    print(k, round(np.trace(np.linalg.matrix_power(Bm, k))), round(float(np.real(np.sum(roots**k))) + (m - n) * (1 + (-1)**k)), end="; ")
print()

u =   0.1: det(I - uB) = 0.9997396779   Bass 0.9997396779
u =   0.3: det(I - uB) = 0.9256746338   Bass 0.9256746338
u = -0.45: det(I - uB) = 1.2695802099   Bass 1.2695802099
u =   0.6: det(I - uB) = -1.1005540493   Bass -1.1005540493
1 0 0; 2 0 0; 3 0 0; 4 0 0; 5 120 120; 6 120 120; 7 0 0; 8 240 240; 9 360 360; 10 1320 1320; 


In [7]:
def mobius(k):
    res, x, p = 1, k, 2
    while p * p <= x:
        if x % p == 0:
            x //= p
            if x % p == 0: return 0
            res = -res
        p += 1
    return -res if x > 1 else res

n = 300; G = nx.random_regular_graph(3, n, seed=2009); m = 3 * n // 2; q = 2
lam = np.sort(np.linalg.eigvalsh(adjacency(G)))
print(f"largest non-trivial |eigenvalue| {max(lam[-2], -lam[0]):.5f}, 2 sqrt 2 = {2 * math.sqrt(2):.5f}")
roots = np.concatenate([np.roots([1, -l, q]) for l in lam])
N = {k: int(round(float(np.real(np.sum(roots**k))))) + (m - n) * (1 + (-1)**k) for k in range(1, 41)}
for k in (5, 10, 15, 20, 25, 30, 35, 40):
    pik = sum(mobius(k // l) * N[l] for l in range(1, k + 1) if k % l == 0) // k
    print(f"k = {k:>2}: pi(k) = {pik:>14}   2^k/k = {2**k / k:>16.1f}   ratio {pik / (2**k / k):.6f}   "
          f"|error| {abs(pik - 2**k / k):.3g} <= bound {6 * n * 2**(k / 2) / k:.3g}")

largest non-trivial |eigenvalue| 2.82310, 2 sqrt 2 = 2.82843
k =  5: pi(k) =              8   2^k/k =              6.4   ratio 1.250000   |error| 1.6 <= bound 2.04e+03
k = 10: pi(k) =            120   2^k/k =            102.4   ratio 1.171875   |error| 17.6 <= bound 5.76e+03
k = 15: pi(k) =           2088   2^k/k =           2184.5   ratio 0.955811   |error| 96.5 <= bound 2.17e+04
k = 20: pi(k) =          52490   2^k/k =          52428.8   ratio 1.001167   |error| 61.2 <= bound 9.22e+04
k = 25: pi(k) =        1342706   2^k/k =        1342177.3   ratio 1.000394   |error| 529 <= bound 4.17e+05
k = 30: pi(k) =       35775642   2^k/k =       35791394.1   ratio 0.999560   |error| 1.58e+04 <= bound 1.97e+06
k = 35: pi(k) =      981669764   2^k/k =      981706810.5   ratio 0.999962   |error| 3.7e+04 <= bound 9.53e+06
k = 40: pi(k) =    27488206540   2^k/k =    27487790694.4   ratio 1.000015   |error| 4.16e+05 <= bound 4.72e+07


Bass's formula holds to all printed digits, and the traces of $B^k$ match the eigenvalue formula exactly (the Petersen graph has girth $5$, so $N_1=\dots=N_4=0$; its $12$ pentagons, run in $2$ directions from $5$ starting edges, give $N_5=120$). The random $3$-regular graph is Ramanujan (as Figure 9.1 showed), and $\pi(k)$ follows $\frac{2^k}k$, with relative error below $0.1\%$ from $k=20$ on. The actual error is far smaller than the bound $6n\,2^{k/2}/k$ of Theorem 9.11: it is of the order of $2^{k/2}$ itself (at $k=40$, $4\times10^5$ against $2^{20}\approx10^6$), because the $n$ terms $\mu_j^k+\mu_j'^k$, of modulus $2\cdot2^{k/2}$, have pseudo-random phases and mostly cancel; the bound adds their moduli. The same is believed of $\zeta$: under the Riemann hypothesis the sum over zeros in the explicit formula is of size about $\sqrt x$, smaller than the trivial bound obtained by adding the moduli of its terms.

### P9.6

**How to find it.** (a) Enumerate $\mathrm{SL}_2(\mathbf F_p)$ by its four entries, index the elements, and build the sparse adjacency matrix; `eigsh` gives the extreme eigenvalues. (b) Take $i$ with $i^2\equiv-1\pmod{13}$ ($i=5$), map $(a_0,a_1,a_2,a_3)$ to $\bigl(\begin{smallmatrix}a_0+ia_1&a_2+ia_3\\-a_2+ia_3&a_0-ia_1\end{smallmatrix}\bigr)$, and work in $\mathrm{PGL}_2$ by normalising each matrix (scale so that its first non-zero entry is $1$); explore the group by breadth-first search from the identity.

In [8]:
def cayley_sl2(p):
    els = [(a, b, c, d) for a in range(p) for b in range(p) for c in range(p) for d in range(p) if (a * d - b * c) % p == 1]
    idx = {g: i for i, g in enumerate(els)}
    gens = [(1, 1, 0, 1), (1, p - 1, 0, 1), (1, 0, 1, 1), (1, 0, p - 1, 1)]
    rows, cols = [], []
    for i, (a, b, c, d) in enumerate(els):
        for (e, f, g, h) in gens:
            rows.append(i); cols.append(idx[((a * e + b * g) % p, (a * f + b * h) % p, (c * e + d * g) % p, (c * f + d * h) % p)])
    return sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(els), len(els)))

for p in (5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53):
    A = cayley_sl2(p)
    top = np.sort(spl.eigsh(A, k=2, which='LA', return_eigenvectors=False))
    bottom = spl.eigsh(A, k=1, which='SA', return_eigenvectors=False)[0]
    print(f"p = {p:>2}: {A.shape[0]:>6} vertices, lambda_2 = {top[0]:.4f}, lambda_min = {bottom:.4f}   (2 sqrt 3 = {2 * math.sqrt(3):.4f})")

p =  5:    120 vertices, lambda_2 = 3.2361, lambda_min = -3.2361   (2 sqrt 3 = 3.4641)
p =  7:    336 vertices, lambda_2 = 3.4142, lambda_min = -3.5616   (2 sqrt 3 = 3.4641)
p = 11:   1320 vertices, lambda_2 = 3.6180, lambda_min = -3.7321   (2 sqrt 3 = 3.4641)
p = 13:   2184 vertices, lambda_2 = 3.6751, lambda_min = -3.8256   (2 sqrt 3 = 3.4641)
p = 17:   4896 vertices, lambda_2 = 3.7093, lambda_min = -3.7453   (2 sqrt 3 = 3.4641)
p = 19:   6840 vertices, lambda_2 = 3.7546, lambda_min = -3.8060   (2 sqrt 3 = 3.4641)


p = 23:  12144 vertices, lambda_2 = 3.7933, lambda_min = -3.8214   (2 sqrt 3 = 3.4641)


p = 29:  24360 vertices, lambda_2 = 3.8178, lambda_min = -3.8482   (2 sqrt 3 = 3.4641)


p = 31:  29760 vertices, lambda_2 = 3.7727, lambda_min = -3.8197   (2 sqrt 3 = 3.4641)


p = 37:  50616 vertices, lambda_2 = 3.8292, lambda_min = -3.8674   (2 sqrt 3 = 3.4641)


p = 41:  68880 vertices, lambda_2 = 3.8191, lambda_min = -3.7982   (2 sqrt 3 = 3.4641)


p = 43:  79464 vertices, lambda_2 = 3.8338, lambda_min = -3.8308   (2 sqrt 3 = 3.4641)


p = 47: 103776 vertices, lambda_2 = 3.8193, lambda_min = -3.8318   (2 sqrt 3 = 3.4641)


p = 53: 148824 vertices, lambda_2 = 3.8256, lambda_min = -3.8348   (2 sqrt 3 = 3.4641)


In [9]:
def lps(p, ell):
    i = next(x for x in range(ell) if (x * x + 1) % ell == 0)
    ev_ = [x for x in range(-p, p + 1) if x % 2 == 0]
    sols = [(a0, a1, a2, a3) for a0 in range(1, int(p**0.5) + 1, 2) for a1 in ev_ for a2 in ev_ for a3 in ev_
            if a0**2 + a1**2 + a2**2 + a3**2 == p]
    gens = [((a0 + i * a1) % ell, (a2 + i * a3) % ell, (-a2 + i * a3) % ell, (a0 - i * a1) % ell) for a0, a1, a2, a3 in sols]
    def norm(g):
        s = next(x for x in g if x % ell)
        inv = pow(s, -1, ell)
        return tuple(x * inv % ell for x in g)
    def mul(g, h):
        a, b, c, d = g; e, f, gg, hh = h
        return norm(((a * e + b * gg) % ell, (a * f + b * hh) % ell, (c * e + d * gg) % ell, (c * f + d * hh) % ell))
    start = norm((1, 0, 0, 1)); idx = {start: 0}; queue = [start]; edges = []
    while queue:
        g = queue.pop()
        for s in gens:
            h = mul(g, s)
            if h not in idx: idx[h] = len(idx); queue.append(h)
            edges.append((idx[g], idx[h]))
    A = np.zeros((len(idx), len(idx)))
    for a, b in edges: A[a, b] += 1
    return len(sols), A

k, A = lps(5, 13)
ev = np.sort(np.linalg.eigvalsh(A))
print(f"{k} generators, {A.shape[0]} vertices (|PGL_2(F_13)| = {13 * (13**2 - 1)}), symmetric: {np.allclose(A, A.T)}")
print(f"extreme eigenvalues {ev[0]:.4f}, {ev[-1]:.4f}; largest non-trivial |eigenvalue| {max(abs(ev[1]), abs(ev[-2])):.4f}; 2 sqrt 5 = {2 * math.sqrt(5):.4f}")

6 generators, 2184 vertices (|PGL_2(F_13)| = 2184), symmetric: True
extreme eigenvalues -6.0000, 6.0000; largest non-trivial |eigenvalue| 4.2497; 2 sqrt 5 = 4.4721


(a) The second eigenvalue grows with $p$ for small $p$, then stabilises around $3.83$: a uniform spectral gap of about $0.17$, as Selberg's $\frac3{16}$ theorem guarantees, but well above the Ramanujan bound $2\sqrt3\approx3.46$. These graphs are expanders, not Ramanujan graphs. (b) The LPS construction needs only the six representations of $5$ as a sum of four squares of the required shape; the generators are closed under inverses (the conjugate quaternion), the graph is connected on all $2184$ elements of $\mathrm{PGL}_2(\mathbf F_{13})$, bipartite (eigenvalue $-6$; here $5$ is not a square modulo $13$), and every other eigenvalue is at most $2\sqrt5$ in absolute value, as the Ramanujan conjecture promises. Sarnak's book *Some Applications of Modular Forms* (Cambridge, 1990) and the book of Davidoff, Sarnak and Valette give the proof.